In [72]:
from datetime import datetime, timedelta
latest_date = datetime.now()
earliest_date = (latest_date - timedelta(days=64)).strftime(r"%Y-%m-%d")
current_date = latest_date.strftime(r"%Y-%m-%d")

In [73]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": 47.52271,
	"longitude": 7.64511,
	"hourly": ["temperature_2m", "relative_humidity_2m", "precipitation", "precipitation_probability", "weather_code", "wind_speed_10m", "cloud_cover", "surface_pressure", "dew_point_2m", "pressure_msl"],
	"timezone": "Europe/Berlin",
	"start_date": earliest_date,
	"end_date": current_date,
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(2).ValuesAsNumpy()
hourly_precipitation_probability = hourly.Variables(3).ValuesAsNumpy()
hourly_weather_code = hourly.Variables(4).ValuesAsNumpy()
hourly_wind_speed_10m = hourly.Variables(5).ValuesAsNumpy()
hourly_cloud_cover = hourly.Variables(6).ValuesAsNumpy()
hourly_surface_pressure = hourly.Variables(7).ValuesAsNumpy()
hourly_dew_point_2m = hourly.Variables(8).ValuesAsNumpy()
hourly_pressure_msl = hourly.Variables(9).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	end =  pd.to_datetime(hourly.TimeEnd() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
hourly_data["precipitation"] = hourly_precipitation
hourly_data["precipitation_probability"] = hourly_precipitation_probability
hourly_data["weather_code"] = hourly_weather_code
hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
hourly_data["cloud_cover"] = hourly_cloud_cover
hourly_data["surface_pressure"] = hourly_surface_pressure
hourly_data["dew_point_2m"] = hourly_dew_point_2m
hourly_data["pressure_msl"] = hourly_pressure_msl

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)
print(hourly_dataframe.info())

Coordinates: 47.52000045776367°N 7.639999866485596°E
Elevation: 293.0 m asl
Timezone: b'Europe/Berlin'b'GMT+2'
Timezone difference to GMT+0: 7200s

Hourly data
                           date  temperature_2m  relative_humidity_2m  \
0    2026-01-31 00:00:00+00:00        0.091000                  94.0   
1    2026-01-31 01:00:00+00:00        0.141000                  94.0   
2    2026-01-31 02:00:00+00:00        0.391000                  94.0   
3    2026-01-31 03:00:00+00:00        0.541000                  94.0   
4    2026-01-31 04:00:00+00:00        0.641000                  94.0   
...                        ...             ...                   ...   
1555 2026-04-05 19:00:00+00:00       20.806499                  38.0   
1556 2026-04-05 20:00:00+00:00       18.706501                  42.0   
1557 2026-04-05 21:00:00+00:00       16.706501                  51.0   
1558 2026-04-05 22:00:00+00:00       15.256500                  61.0   
1559 2026-04-05 23:00:00+00:00       14.456500 

In [68]:
###
# Prepare Hopsworks project

import hopsworks
from dotenv import load_dotenv
import os

env = load_dotenv(".env")

project = hopsworks.login(
    project='mlops',  # Replace with your project name
    host="eu-west.cloud.hopsworks.ai",
    port=443,
    api_key_value=os.environ["HOPSWORKS_API_KEY"]  # Get from Hopsworks UI > Account Settings > API Keys
)

fs = project.get_feature_store()


2026-04-05 20:52:11,347 INFO: Closing external client and cleaning up certificates.
2026-04-05 20:52:11,387 INFO: Connection closed.
2026-04-05 20:52:11,388 INFO: Initializing external client
2026-04-05 20:52:11,388 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-04-05 20:52:12,700 INFO: Python Engine initialized.

Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/31926


In [80]:
# Feature Preparation and cleanup

feature_dataframe = hourly_dataframe.copy()

# Weather Variables could depend on the time of day
feature_dataframe["derived_hour"] = feature_dataframe["date"].apply(lambda x: x.hour)
# Weather Variables could depend on the month of the year
feature_dataframe["derived_month"] = feature_dataframe["date"].apply(lambda x: x.month)
# Generate Partion Key
feature_dataframe["partition"] = feature_dataframe["date"].apply(lambda x: f'{x.year}-{x.month}')
# Rain Probability could depend on the average humidity of the last 24 hours
feature_dataframe["timestamp"] = feature_dataframe["date"] # Copy to save column
feature_dataframe.set_index('date', inplace=True) # Needed for averaging over time later
feature_dataframe['relative_humidity_2m_24h_avg'] = feature_dataframe['relative_humidity_2m'].rolling('24h').mean()
# Rain Probability could depend on if the barometric pressure is rising or falling
feature_dataframe['pressure_msl_3h_pct_change'] = feature_dataframe['pressure_msl'].pct_change(periods=3)

# Find first valid index (during preparation pressure_msl_3h_pct_change returned the latest valid row)
first_non_nan_index = feature_dataframe['pressure_msl_3h_pct_change'].first_valid_index()

# Slice the DataFrame to keep only rows from the first non-NaN index onward
feature_dataframe = feature_dataframe.loc[first_non_nan_index:]


print(feature_dataframe.info())
print(feature_dataframe.head())
print("min_timesamp", feature_dataframe["timestamp"].min())
print("max_timestamp", feature_dataframe["timestamp"].max())



<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1557 entries, 2026-01-31 03:00:00+00:00 to 2026-04-05 23:00:00+00:00
Data columns (total 16 columns):
 #   Column                        Non-Null Count  Dtype              
---  ------                        --------------  -----              
 0   temperature_2m                1557 non-null   float32            
 1   relative_humidity_2m          1557 non-null   float32            
 2   precipitation                 1557 non-null   float32            
 3   precipitation_probability     1557 non-null   float32            
 4   weather_code                  1557 non-null   float32            
 5   wind_speed_10m                1557 non-null   float32            
 6   cloud_cover                   1557 non-null   float32            
 7   surface_pressure              1557 non-null   float32            
 8   dew_point_2m                  1557 non-null   float32            
 9   pressure_msl                  1557 non-null   float32      

In [ ]:
fg = fs.create_feature_group("weather_prediction", version=1, primary_key=["timestamp"], event_time="timestamp", partition_key=["partition"], online_enabled=True)
fg.save(feature_dataframe)

Feature Group created successfully, explore it at 
https://eu-west.cloud.hopsworks.ai:443/p/31926/fs/20610/fg/37818


Uploading Dataframe: 100.00% |██████████| Rows 1544/1544 | Elapsed Time: 00:00 | Remaining Time: 00:00


Launching job: weather_prediction_1_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://eu-west.cloud.hopsworks.ai:443/p/31926/jobs/named/weather_prediction_1_offline_fg_materialization/executions


(Job('weather_prediction_1_offline_fg_materialization', 'PYSPARK'), None)

In [83]:
# Concatenate with previous DataFrame (if available)
## Get most current previous feature group as basis to concatenate

feature_group_name = "weather_prediction"

try:
    # Get all versions of the feature group
    fgs = fs.get_feature_groups()
    fg_versions = [fg for fg in fgs if fg.name == feature_group_name]

    if not fg_versions:
        print(f"Feature group '{feature_group_name}' does not exist.")
    else:
        # Get the latest version
        latest_version = max(fg_versions, key=lambda x: x.version).version
        fg = fs.get_feature_group(feature_group_name, version=latest_version)

        # Read the data into a pandas DataFrame
        old_feature_dataframe = fg.read()
        print(f"Data loaded successfully for version {latest_version}.")
        print(old_feature_dataframe.info())
except Exception as e:
    print(f"An error occurred: {e}")






Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (1.79s) 
Data loaded successfully for version 1.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1544 entries, 0 to 1543
Data columns (total 16 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   partition                     1544 non-null   object        
 1   temperature_2m                1544 non-null   float32       
 2   relative_humidity_2m          1544 non-null   float32       
 3   precipitation                 1544 non-null   float32       
 4   precipitation_probability     1544 non-null   float32       
 5   weather_code                  1544 non-null   float32       
 6   wind_speed_10m                1544 non-null   float32       
 7   cloud_cover                   1544 non-null   float32       
 8   surface_pressure              1544 non-null   float32       
 9   dew_point_2m                  1544 n

In [84]:
## Add new rows (timestamp > max existing timestamp)
## Does not handle switch between Daylight Savings Time but these are only few samples and 
## can be neglected since model Quality is not relevant for project
old_feature_dataframe['timestamp'] = old_feature_dataframe['timestamp'].dt.tz_localize('Europe/Berlin', nonexistent="shift_forward")
latest_timestamp = old_feature_dataframe['timestamp'].max()
new_rows = feature_dataframe[feature_dataframe['timestamp'] > latest_timestamp]
new_feature_dataframe: pd.DataFrame = pd.concat([old_feature_dataframe, new_rows], ignore_index=True).sort_values('timestamp')
new_feature_dataframe.sort_values("timestamp")


,partition,temperature_2m,relative_humidity_2m,precipitation,precipitation_probability,weather_code,wind_speed_10m,cloud_cover,surface_pressure,dew_point_2m,pressure_msl,derived_hour,derived_month,timestamp,relative_humidity_2m_24h_avg,pressure_msl_3h_pct_change
1488,2026-1,-1.080500,81.0,0.0,0.0,3.0,13.104198,100.0,999.223206,-3.926583,1004.000000,16,1,2026-01-29 16:00:00+01:00,81.000000,0.000000
1489,2026-1,-1.230500,82.0,0.0,0.0,3.0,13.207634,100.0,999.319824,-3.909390,1004.099976,17,1,2026-01-29 17:00:00+01:00,81.200000,-0.000100
1490,2026-1,-1.480500,82.0,0.0,0.0,3.0,13.104198,100.0,999.813049,-4.153884,1004.599976,18,1,2026-01-29 18:00:00+01:00,81.333333,0.000697
1491,2026-1,-1.780500,83.0,0.0,0.0,3.0,12.641076,100.0,1000.006775,-4.286089,1004.799988,19,1,2026-01-29 19:00:00+01:00,81.571429,0.000797
1492,2026-1,-2.330500,86.0,0.0,0.0,3.0,12.641076,100.0,1000.096619,-4.353434,1004.900024,20,1,2026-01-29 20:00:00+01:00,82.125000,0.000797
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1589,2026-4,20.806499,38.0,0.0,0.0,3.0,13.324863,100.0,986.250610,5.966179,1020.299988,19,4,2026-04-05 19:00:00+00:00,68.958333,0.000196
1590,2026-4,18.706501,42.0,0.0,0.0,3.0,10.041354,85.0,986.976990,5.530221,1021.299988,20,4,2026-04-05 20:00:00+00:00,67.916667,0.000490
1591,2026-4,16.706501,51.0,0.0,0.0,3.0,7.517021,86.0,988.000793,6.517068,1022.599976,21,4,2026-04-05 21:00:00+00:00,67.041667,0.002647
1592,2026-4,15.256500,61.0,0.0,3.0,61.0,9.290511,100.0,988.893127,7.780620,1023.700012,22,4,2026-04-05 22:00:00+00:00,66.125000,0.003332


In [90]:
# Plausibility Checks before Upload
# Due to selected datasource and data preparation, this can only be updated once per day
print(new_feature_dataframe["timestamp"].min())
print(new_feature_dataframe["timestamp"].max())
assert new_feature_dataframe["timestamp"].min() == old_feature_dataframe["timestamp"].min()
assert new_feature_dataframe["timestamp"].max() > old_feature_dataframe["timestamp"].max()

# Upload updated feature dataframe to hopsworks
new_version = latest_version + 1 # TODO Think if the latest date would be better


new_fg = fs.create_feature_group("weather_prediction", version=new_version, primary_key=["timestamp"], event_time="timestamp", partition_key=["partition"], online_enabled=True)
new_fg.save(feature_dataframe)

2026-01-29 16:00:00+01:00
2026-04-05 23:00:00+00:00


RestAPIError: Metadata operation error: (url: https://eu-west.cloud.hopsworks.ai/hopsworks-api/api/project/31926/featurestores/20610/featuregroups). Server response: 
HTTP code: 400, HTTP reason: Bad Request, body: b'{"errorCode":270089,"usrMsg":"project: mlops, featurestoreId: 20610","errorMsg":"The feature group you are trying to create does already exist."}', error code: 270089, error msg: The feature group you are trying to create does already exist., user msg: project: mlops, featurestoreId: 20610

Feature Group created successfully, explore it at 
https://eu-west.cloud.hopsworks.ai:443/p/31926/fs/20610/fg/37842


Uploading Dataframe: 100.00% |██████████| Rows 1557/1557 | Elapsed Time: 00:00 | Remaining Time: 00:00


Launching job: weather_prediction_2_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://eu-west.cloud.hopsworks.ai:443/p/31926/jobs/named/weather_prediction_2_offline_fg_materialization/executions


(Job('weather_prediction_2_offline_fg_materialization', 'PYSPARK'), None)